# 5.03 Self-Query Retriever

`SelfQueryRetriever`는 **자연어 질의를 두 부분으로 분해**한다.
- semantic query — 실제 임베딩 유사도 검색에 쓰는 텍스트
- metadata filter — LLM이 추출한 구조적 조건 (e.g. `year >= 2024`, `rating > 8.5`, `genre == "drama"`)

LLM이 추출한 filter를 **vector store DSL**(Chroma, Pinecone, Qdrant 등)로 번역해 백엔드에 붙여 보낸다.

## 학습 목표

- `AttributeInfo(name, description, type)` 리스트로 **필터 가능한 metadata 스키마**를 LLM에 알린다
- `SelfQueryRetriever.from_llm(llm, vectorstore, document_contents, metadata_field_info=...)` 로 조립
- `enable_limit=True` 로 "가장 최근 3개" 같은 Top-K 추출도 위임
- `create_retriever_tool`로 에이전트에 물린다

## 언제 쓰나

- 사용자 질의에 **"2023년 이후"**, **"평점 9 이상"** 같은 **정량 조건**이 섞일 때
- 프런트엔드에 필터 UI 없이 **한 줄 검색**으로 정형 조건을 처리하고 싶을 때
- 문서에 태그·카테고리·날짜 같은 구조 메타데이터가 **이미 정돈**돼 있을 때


## 5.03.1 환경 설정

Self-Query는 metadata filter를 벡터 스토어 DSL로 번역해야 해서 **지원 백엔드**가 필요하다. 이 노트북은 `langchain-chroma`(로컬)의 `ChromaTranslator`를 쓴다.
필터 파서는 `lark`에 의존한다.


In [ ]:
# !pip install -U langchain langchain-classic langchain-core langchain-openai langchain-chroma lark

import os
from dotenv import load_dotenv
load_dotenv(override=True)

assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY 필요 (LLM + embedding)"

import lark  # noqa: F401  # filter grammar parser


## 5.03.2 기본 사용법 — 영화 메타데이터 예제

자주 쓰이는 교본 예제: 영화 설명 + `year`, `rating`, `genre`, `director` 4개 metadata.
자연어 질의 하나가 들어오면 `SelfQueryRetriever`가 내부적으로:
1. LLM에게 **구조적 query 생성**을 시킴 (LangChain 내장 query_constructor chain)
2. 결과를 **ChromaTranslator**가 Chroma 필터 dict(`$and`/`$gt`/`$eq` …)로 번역
3. Chroma에 `similarity_search(query, filter=...)` 호출


In [ ]:
from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain.chat_models import init_chat_model
from langchain_classic.chains.query_constructor.schema import AttributeInfo
from langchain_classic.retrievers.self_query.base import SelfQueryRetriever

movies = [
    Document(
        page_content="고독한 해커가 가상 현실을 탈출하는 디스토피아 스릴러",
        metadata={"year": 1999, "rating": 8.7, "genre": "action", "director": "Wachowski"},
    ),
    Document(
        page_content="꿈 속의 꿈을 설계해 기업 비밀을 훔치는 도둑 이야기",
        metadata={"year": 2010, "rating": 8.8, "genre": "sci-fi", "director": "Nolan"},
    ),
    Document(
        page_content="농경 위기의 지구를 떠나 다른 은하로 향하는 우주 여행",
        metadata={"year": 2014, "rating": 8.6, "genre": "sci-fi", "director": "Nolan"},
    ),
    Document(
        page_content="고립된 호텔에서 광기에 휩싸이는 작가",
        metadata={"year": 1980, "rating": 8.4, "genre": "horror", "director": "Kubrick"},
    ),
    Document(
        page_content="블루칼라 동네에서 복싱으로 삶을 바꿔 가는 이야기",
        metadata={"year": 1976, "rating": 8.1, "genre": "drama", "director": "Avildsen"},
    ),
    Document(
        page_content="실크로드 사막을 배경으로 펼쳐지는 첩보 액션",
        metadata={"year": 2023, "rating": 7.9, "genre": "action", "director": "Park"},
    ),
]

vectorstore = Chroma.from_documents(
    movies,
    embedding=OpenAIEmbeddings(model="text-embedding-3-small"),
    collection_name="self_query_movies",
)

metadata_field_info = [
    AttributeInfo(name="year",     description="개봉 연도 (정수)",            type="integer"),
    AttributeInfo(name="rating",   description="IMDb 평점 (0-10 실수)",       type="float"),
    AttributeInfo(name="genre",    description="영화 장르 (action/sci-fi/horror/drama)", type="string"),
    AttributeInfo(name="director", description="감독 성",                     type="string"),
]

llm = init_chat_model("openai:gpt-4.1", temperature=0)  # self-query는 구조화 출력에 민감, mini보다 안정적

retriever = SelfQueryRetriever.from_llm(
    llm=llm,
    vectorstore=vectorstore,
    document_contents="영화 한 줄 줄거리",
    metadata_field_info=metadata_field_info,
)

for d in retriever.invoke("Nolan 감독의 SF 영화"):
    print("-", d.metadata.get("year"), d.metadata.get("genre"), "|", d.page_content[:40])


## 5.03.3 주요 옵션

| 옵션 | 의미 |
|------|------|
| `enable_limit=True` | 질의에 "상위 3개" 같은 구절이 있으면 LLM이 `limit`까지 추출해 반환 개수를 동적으로 정한다 |
| `use_original_query=True` | filter 추출 후에도 **원문 그대로** semantic query로 쓴다 — LLM이 질의 자체를 바꾸는 걸 막고 싶을 때 |
| `search_kwargs={"k": ...}` | 베이스 vectorstore에 전달할 기본 검색 파라미터 |
| `verbose=True` | 내부적으로 만든 structured query를 출력 (디버깅) |

### enable_limit 예


In [ ]:
retriever_limited = SelfQueryRetriever.from_llm(
    llm=llm,
    vectorstore=vectorstore,
    document_contents="영화 한 줄 줄거리",
    metadata_field_info=metadata_field_info,
    enable_limit=True,
    verbose=True,
)

hits = retriever_limited.invoke("평점 8.5 이상 SF 2개만 보여줘")
print("count:", len(hits))
for d in hits:
    print("-", d.metadata, "|", d.page_content[:40])


## 5.03.4 결과 비교 — 필터 있/없

같은 백엔드(Chroma)에서 **필터가 동작했는가** 확인하려면 순수 `similarity_search(query)`와 직접 비교해 본다.
자연어 조건이 어휘적으로 반영되지 않을 때 — 예: `"2020년 이후 액션"` — 순수 벡터 검색은 옛날 액션 영화도 뽑지만, self-query는 `year >= 2020 AND genre == 'action'`을 DSL로 붙여 정확히 거른다.


In [ ]:
query = "2020년 이후 액션"

print("[순수 similarity_search — 필터 없음]")
for d in vectorstore.similarity_search(query, k=4):
    print("-", d.metadata.get("year"), d.metadata.get("genre"), "|", d.page_content[:30])

print("\n[SelfQueryRetriever — LLM이 자동 필터 주입]")
for d in retriever.invoke(query):
    print("-", d.metadata.get("year"), d.metadata.get("genre"), "|", d.page_content[:30])


## 5.03.5 `create_agent` 연동

Self-query retriever도 다른 retriever와 마찬가지로 `create_retriever_tool(...)`로 감싸 에이전트 도구로 노출한다. 에이전트는 **도구 설명**만 보고 호출하므로, `description`에 "정량 조건도 자연어로 받는다"라고 명시해 주는 것이 도움이 된다.


In [ ]:
from langchain_classic.tools.retriever import create_retriever_tool
from langchain.agents import create_agent

movie_tool = create_retriever_tool(
    retriever,
    name="search_movies",
    description=(
        "영화 DB를 검색한다. 자연어 질의에 연도/평점/장르/감독 같은 정량·범주 조건이 섞여도 "
        "자동으로 metadata 필터를 만들어 적용한다."
    ),
)

agent = create_agent(
    model="openai:gpt-4.1-mini",
    tools=[movie_tool],
    system_prompt="사용자 요청을 search_movies 도구로 먼저 조회하고 결과를 기반으로 답하라.",
)

r = agent.invoke({"messages": [{"role": "user", "content": "Kubrick이 감독한 평점 8 이상 영화 추천해줘"}]})
print(r["messages"][-1].content)


## 체크리스트

- [ ] `AttributeInfo.description`은 LLM이 filter를 조립할 때 결정적 입력 — "숫자이다" 수준이 아니라 **의미**까지 한 줄로 적기
- [ ] `type`은 `"string" | "integer" | "float"` 만 지원 (list/dict는 처리 불가). boolean은 string으로 인코딩
- [ ] `SelfQueryRetriever`는 **지원 백엔드** 필요: Chroma / Pinecone / Qdrant / PGVector / Weaviate / Milvus / Elasticsearch 등 — translator 모듈이 존재해야 한다
- [ ] LLM 비용이 질의당 발생 — 고빈도 앱은 쿼리 캐시·결과 캐시를 앞단에 두기
- [ ] filter 추출이 실패해도 **예외**는 잘 안 나지만 **빈 filter**로 전체 검색이 될 수 있음 → `verbose=True`로 한 번은 프롬프트 디버깅

## 다음

- `04_web_wiki_arxiv.ipynb` — 외부 지식 소스(Tavily · Wikipedia · arXiv · PubMed)
- `03_vectorstores/02_chroma.ipynb` — Chroma 백엔드 직접 다루기

## 참고

- 공식 How-to: https://python.langchain.com/docs/how_to/self_query/
- 지원 translator 목록: `langchain_classic.retrievers.self_query.*`
